# Milestone 1 — Accommodation Market Demand & Supply Analysis
**Nama:** Nawafi Suren  
**Program:** Hacktiv8 Data Science Fulltime Program  
**Status dataset:** DATA DUMMY / SIMULASI UNTUK BELAJAR — BUKAN DATA RIIL DAN BUKAN UNTUK SUBMISSION FINAL.

> Notebook ini dibuat untuk mempelajari alur MS1 dari awal sampai akhir. Instruksi resmi MS1 mensyaratkan dataset non-dummy yang sumbernya dapat dipertanggungjawabkan. Ganti dataset simulasi ini dengan dataset asli yang sudah disetujui buddy sebelum submission.

## 1. Introduction
Project ini mencontohkan analisis pasar penginapan dengan sudut pandang pemilik atau calon investor akomodasi. Fokusnya adalah membandingkan **supply** (jumlah listing) dan **indikasi aktivitas tamu** (review), lalu melihat perbedaan berdasarkan lokasi, tipe kamar, dan harga.

**Pertanyaan utama:** Lokasi dan tipe penginapan mana yang menunjukkan indikasi aktivitas tamu relatif tinggi dibandingkan jumlah listing?

**Batasan penting:** `reviews_per_month` adalah rata-rata review bulanan, bukan jumlah booking atau okupansi. Hasil hanya menunjukkan proxy aktivitas tamu.

## 2. Problem Statement and Dataset Description

### 2.1 Latar belakang
Pemilik penginapan perlu memahami kondisi pasar sebelum menentukan lokasi, tipe properti, dan strategi harga. Jumlah listing menunjukkan banyaknya penawaran, sedangkan review dapat digunakan sebagai indikator tidak langsung aktivitas tamu. Membaca keduanya bersama-sama membantu menghindari kesimpulan keliru bahwa lokasi dengan listing terbanyak otomatis memiliki demand tertinggi.

### 2.2 SMART problem statement
- **Specific:** Mengidentifikasi wilayah dan tipe penginapan yang memiliki indikasi aktivitas tamu relatif tinggi.
- **Measurable:** Menggunakan jumlah listing, `reviews_per_month`, harga, dan room type.
- **Achievable:** Menganalisis dataset listing menggunakan Python, Pandas, visualisasi, serta uji statistik.
- **Relevant:** Memberikan gambaran awal bagi pemilik penginapan dalam memahami supply, aktivitas tamu, dan posisi harga.
- **Time-bound:** Analisis dilakukan pada snapshot dataset simulasi ini. Pada project nyata, periode mengikuti tanggal snapshot dataset asli.

### 2.3 Penjabaran masalah (5W+1H)
1. Wilayah mana yang memiliki jumlah listing paling banyak?
2. Wilayah mana yang memiliki total aktivitas review bulanan tertinggi?
3. Wilayah mana yang memiliki rata-rata review bulanan per listing tertinggi?
4. Bagaimana hubungan harga dengan aktivitas review bulanan?
5. Bagaimana distribusi harga dan aktivitas review?
6. Apakah aktivitas review bulanan berbeda antar-room type?

### 2.4 Dataset
File: `data/P0M1_nawafi_suren_dataset.csv`  
Jumlah baris: 600 listing simulasi.  
Sumber: dibuat secara sintetis untuk latihan; tidak berasal dari Airbnb atau Inside Airbnb.

### 2.5 Data dictionary
| Kolom | Arti |
|---|---|
| `id` | ID listing simulasi |
| `name` | Nama listing simulasi |
| `neighbourhood` | Wilayah |
| `room_type` | Tipe penginapan |
| `price` | Harga per malam (USD simulasi) |
| `minimum_nights` | Minimum malam menginap |
| `number_of_reviews` | Total review simulasi |
| `reviews_per_month` | Rata-rata review per bulan simulasi |
| `availability_365` | Hari tersedia dalam 365 hari |
| `host_listings_count` | Jumlah listing yang dimiliki host |

### 2.6 Catatan interpretasi
- Total listing = proxy supply / jumlah penawaran yang tercatat.
- `reviews_per_month` = proxy aktivitas tamu, bukan booking.
- Rata-rata review per listing = indikator relatif untuk membandingkan wilayah dengan ukuran supply berbeda.
- `availability_365` tidak otomatis berarti kamar kosong atau sudah dipesan; kalender dapat dipengaruhi blokir host.

## 3. Data Loading

Pertama, import library. Jika library belum tersedia, jalankan `pip install -r requirements.txt` di terminal atau environment notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

# Path ini cocok jika notebook dibuka dari folder utama hasil ekstrak ZIP.
DATA_PATH = Path('data/P0M1_nawafi_suren_dataset.csv')
df_raw = pd.read_csv(DATA_PATH)

print('Ukuran data:', df_raw.shape)
display(df_raw.head())

### Eksplorasi awal
`shape` menunjukkan jumlah baris dan kolom. `info()` membantu melihat tipe data dan nilai non-null. `describe()` menampilkan ringkasan numerik.

In [ ]:
df_raw.info()
display(df_raw.describe(include='all').T)
print('Jumlah duplikat:', df_raw.duplicated().sum())
print('Missing values per kolom:')
display(df_raw.isna().sum())

## 4. Data Cleaning

Tahap ini menyiapkan data sebelum analisis. Pada dataset simulasi ini, cleaning tetap dilakukan agar alur pengerjaan terlihat. Pada dataset asli, jangan menghapus missing value secara otomatis tanpa memahami penyebabnya.

In [ ]:
df = df_raw.copy()

# Rapikan nama kolom
df.columns = df.columns.str.strip().str.lower()

# Hapus baris duplikat
df = df.drop_duplicates()

# Pastikan kolom numerik memiliki tipe angka
numeric_cols = [
    'price', 'minimum_nights', 'number_of_reviews',
    'reviews_per_month', 'availability_365', 'host_listings_count'
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Ubah kolom kategori menjadi string yang rapi
for col in ['neighbourhood', 'room_type']:
    df[col] = df[col].astype('string').str.strip()

# Periksa missing value
display(df.isna().sum())

# Untuk latihan ini, hapus baris yang missing pada kolom inti analisis.
core_cols = ['id', 'neighbourhood', 'room_type', 'price', 'reviews_per_month']
df = df.dropna(subset=core_cols)

# Harga tidak boleh nol atau negatif
df = df[df['price'] > 0]

print('Ukuran data setelah cleaning:', df.shape)

## 5. Exploration and Analysis

### 5.1 Membuat tabel ringkasan per lokasi
Kita hitung jumlah listing, total review bulanan, rata-rata review bulanan per listing, dan rata-rata harga. `groupby()` mengelompokkan data; `agg()` menghitung beberapa metrik.

In [ ]:
location_summary = (
    df.groupby('neighbourhood')
      .agg(
          total_listing=('id', 'count'),
          total_reviews_per_month=('reviews_per_month', 'sum'),
          avg_reviews_per_listing=('reviews_per_month', 'mean'),
          avg_price=('price', 'mean'),
          median_price=('price', 'median')
      )
      .reset_index()
)

location_summary = location_summary.sort_values(
    'avg_reviews_per_listing', ascending=False
)
display(location_summary.round(2))

### Visualisasi 1 — Jumlah listing per lokasi
Grafik ini menjawab: lokasi mana yang memiliki supply listing paling banyak?

In [ ]:
supply = location_summary.sort_values('total_listing', ascending=False)
plt.figure(figsize=(10, 5))
plt.bar(supply['neighbourhood'], supply['total_listing'])
plt.title('Jumlah Listing Penginapan per Lokasi')
plt.xlabel('Lokasi')
plt.ylabel('Jumlah listing')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

**Cara membaca:** batang tertinggi menunjukkan lokasi dengan listing paling banyak. Ini menunjukkan supply, bukan otomatis demand tertinggi.

### Visualisasi 2 — Total aktivitas review bulanan per lokasi

In [ ]:
activity = location_summary.sort_values('total_reviews_per_month', ascending=False)
plt.figure(figsize=(10, 5))
plt.bar(activity['neighbourhood'], activity['total_reviews_per_month'])
plt.title('Total Aktivitas Review Bulanan per Lokasi')
plt.xlabel('Lokasi')
plt.ylabel('Total reviews per month (proxy)')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

**Cara membaca:** total lebih tinggi dapat terjadi karena listing lebih banyak. Karena itu, grafik ini sebaiknya dibaca bersama grafik rata-rata per listing.

### Visualisasi 3 — Rata-rata aktivitas review per listing

In [ ]:
relative = location_summary.sort_values('avg_reviews_per_listing', ascending=False)
plt.figure(figsize=(10, 5))
plt.bar(relative['neighbourhood'], relative['avg_reviews_per_listing'])
plt.title('Rata-rata Review Bulanan per Listing')
plt.xlabel('Lokasi')
plt.ylabel('Review per listing per bulan')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

**Cara membaca:** nilai lebih tinggi menunjukkan aktivitas review relatif lebih tinggi per listing. Ini merupakan proxy, bukan ukuran booking aktual. Untuk menghindari kesimpulan dari sampel kecil, pada data riil sebaiknya tetapkan batas minimum jumlah listing per lokasi.

### Visualisasi 4 — Hubungan harga dan aktivitas review

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df, x='price', y='reviews_per_month',
    hue='room_type', alpha=0.65
)
plt.title('Harga vs Aktivitas Review Bulanan')
plt.xlabel('Harga per malam (USD simulasi)')
plt.ylabel('Reviews per month (proxy)')
plt.tight_layout()
plt.show()

correlation = df[['price', 'reviews_per_month']].corr(method='spearman').iloc[0, 1]
print('Korelasi Spearman harga vs review_per_month:', round(correlation, 3))

**Cara membaca:** perhatikan pola sebaran titik. Korelasi tidak membuktikan sebab-akibat. Harga dapat berkaitan dengan lokasi, ukuran, fasilitas, dan tipe kamar.

### 5.2 Descriptive Statistics
Pertanyaan: bagaimana karakteristik distribusi harga dan aktivitas review?

In [ ]:
descriptive = df[['price', 'reviews_per_month', 'availability_365']].describe().T
descriptive['median'] = df[['price', 'reviews_per_month', 'availability_365']].median()
descriptive['IQR'] = (
    df[['price', 'reviews_per_month', 'availability_365']].quantile(0.75)
    - df[['price', 'reviews_per_month', 'availability_365']].quantile(0.25)
)
display(descriptive.round(2))

**Panduan newbie:**
- Mean: nilai rata-rata.
- Median: nilai tengah setelah data diurutkan.
- Standard deviation (std): ukuran seberapa menyebar data dari mean.
- IQR: Q3 - Q1, rentang 50% data tengah.
- Jika mean jauh lebih besar daripada median, distribusi mungkin memiliki ekor kanan/outlier tinggi.

### 5.3 Inferential Statistics — Kruskal-Wallis
Pertanyaan: apakah aktivitas review bulanan berbeda antar-room type?

Hipotesis:
- H0: distribusi `reviews_per_month` sama pada semua room type.
- H1: setidaknya ada satu room type yang distribusinya berbeda.

Kruskal-Wallis dipilih sebagai contoh uji non-parametrik untuk membandingkan lebih dari dua kelompok independen. Uji ini menguji perbedaan distribusi/ranking, bukan langsung membuktikan perbedaan mean.

In [ ]:
groups = [
    group['reviews_per_month'].dropna().values
    for _, group in df.groupby('room_type')
]

statistic, p_value = stats.kruskal(*groups)

alpha = 0.05
print('H-statistic:', round(statistic, 4))
print('p-value:', p_value)

if p_value < alpha:
    print('Keputusan: Tolak H0. Ada bukti perbedaan distribusi aktivitas review pada setidaknya dua room type.')
else:
    print('Keputusan: Gagal menolak H0. Belum cukup bukti adanya perbedaan distribusi aktivitas review antar-room type.')

**Interpretasi:** bandingkan p-value dengan alpha 0.05. Jika p-value < 0.05, hasil uji signifikan secara statistik. Untuk mengetahui pasangan kelompok mana yang berbeda, perlu uji post-hoc (misalnya Dunn test) dengan koreksi multiple comparisons. Jangan menyatakan satu room type paling diminati hanya dari hasil Kruskal-Wallis.

### 5.4 Ringkasan otomatis untuk membantu interpretasi

In [ ]:
top_supply = location_summary.loc[location_summary['total_listing'].idxmax()]
top_activity = location_summary.loc[location_summary['total_reviews_per_month'].idxmax()]
top_relative = location_summary.loc[location_summary['avg_reviews_per_listing'].idxmax()]

print('Supply listing tertinggi:', top_supply['neighbourhood'],
      '-', int(top_supply['total_listing']), 'listing')
print('Total aktivitas review tertinggi:', top_activity['neighbourhood'],
      '-', round(top_activity['total_reviews_per_month'], 2))
print('Aktivitas review rata-rata per listing tertinggi:', top_relative['neighbourhood'],
      '-', round(top_relative['avg_reviews_per_listing'], 2))

## 6. Conclusion

Gunakan hasil aktual yang muncul ketika notebook dijalankan untuk menulis kesimpulan. Template:

1. **Supply:** lokasi dengan jumlah listing terbesar adalah [isi berdasarkan output].
2. **Aktivitas total:** lokasi dengan total review bulanan terbesar adalah [isi berdasarkan output].
3. **Aktivitas relatif:** lokasi dengan rata-rata review per listing terbesar adalah [isi berdasarkan output].
4. **Harga:** hubungan harga dan aktivitas review berdasarkan korelasi Spearman adalah [isi nilai dan arah hubungan]. Korelasi bukan sebab-akibat.
5. **Statistik inferensial:** berdasarkan p-value Kruskal-Wallis [isi keputusan]. Jika signifikan, lakukan post-hoc sebelum menyebut pasangan room type yang berbeda.
6. **Rekomendasi:** gunakan hasil sebagai screening awal peluang pasar, bukan keputusan investasi final. Validasi dengan data booking/occupancy aktual, biaya operasional, musim, dan kompetisi.

**Kesimpulan metodologis:** dataset latihan ini sintetis. Angka dan insight hanya untuk belajar alur analisis, tidak boleh dipresentasikan sebagai fakta pasar Singapore atau Airbnb.

## Appendix — Checklist sebelum submission MS1

- [ ] Ganti dataset dummy dengan dataset asli non-dummy yang sumbernya dapat dipertanggungjawabkan.
- [ ] Konsultasikan URL dataset, latar belakang, tujuan, dan calon pembaca report kepada buddy.
- [ ] Sesuaikan nama kolom dan seluruh kode dengan dataset asli.
- [ ] Cantumkan URL sumber dataset pada Problem Statement and Dataset Description.
- [ ] Jalankan ulang seluruh cell dari atas ke bawah (Restart & Run All).
- [ ] Buat Tableau Public dashboard: minimal 4 visualisasi dan minimal 2 interaktivitas.
- [ ] Dashboard harus memiliki bagian Data Visualization dan Statistical Analysis.
- [ ] Tempel URL dashboard pada bagian paling atas notebook.
- [ ] Lengkapi `description.md`.
- [ ] Gunakan nama notebook `P0M1_nawafi_suren.ipynb` dan dataset `P0M1_nawafi_suren_dataset.csv`.
- [ ] Jangan mengunggah dataset dummy ini sebagai dataset final tugas karena instruksi resmi MS1 melarang data dummy.